# Correctly classified case — complete comparison across four pipelines

**Default: ISIC_0000013**, the melanoma correctly detected by your saved Attention U-Net–EfficientNet pipeline (previous recorded score approximately **0.3846**, cutoff approximately **0.3088**).

1. Open this notebook in **Google Colab**.
2. Run the **single code cell** and connect the same Drive account used for your completed four-pipeline comparison.
3. A ZIP containing the comparison PNGs downloads automatically. **No retraining is performed.** A GPU is optional.

The image and expert mask are already embedded. You do not need to upload the image or download the dataset. The cell finds the checkpoints from your completed comparison in Drive.

## Images produced

Each row represents **Attention U-Net, U-Net, ResUNet or TransUNet**, using the same selected image. The complete figure includes:

**Original image → Ground-truth mask → Segmentation probability map → Predicted lesion mask → Mask overlay → Isolated lesion → Classifier lesion ROI → Grad-CAM heatmap → Grad-CAM overlay.**

The ZIP contains the full comparison, two larger panels for presentation slides, one full image per pipeline, and every stage as a separate PNG. The predictions table includes the actual diagnosis, each model's prediction, whether it is correct, model score, cutoff, Dice and IoU.

To compare the **correctly identified benign example**, change the selector at the top of the code cell to **ISIC_0000003** and run it again (previous Attention score approximately **0.1694**).

These were selected because Attention classified them correctly. Other pipelines' predictions are calculated and displayed as they are; no prediction or threshold is changed to make it correct. These examples do not establish an overall winner. The source models have different training splits and ROI methods. The uploaded U-Net++ notebook implements a standard U-Net, so it is labelled U-Net.

All Grad-CAM panels explain the **melanoma class**, with the ROI branch used for Attention. Each heatmap is normalised independently and is not a lesion mask. The benign example's Grad-CAM therefore explains melanoma support, even if its predicted class is benign.

Image files, reference masks and diagnosis labels were checked against the official ISIC 2016 Part3B test archive and label CSV. The notebook checks the embedded file hashes before inference. If saved weights are missing, it reports the missing files and stops without training.


In [ ]:
#@title Run the selected correctly classified example through all four saved pipelines
# Select ISIC_0000013 for the melanoma example or ISIC_0000003 for the benign example.
IMAGE_ID = "ISIC_0000013" #@param ["ISIC_0000013", "ISIC_0000003"]

from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import importlib
import sys
import tensorflow as tf
from IPython.display import display, Image as DisplayImage
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
COMPARISON_ROOT = DRIVE_ROOT / 'Melanoma_BestCase_Comparison_From_Notebooks'
PREFERRED_COMPARISON = COMPARISON_ROOT / 'comparisons/20261014T185422_968354Z'
RUNTIME = Path('/content/melanoma_correct_case_comparison')
MODULE_DIR = RUNTIME / 'modules'
CASE_DIR = RUNTIME / 'cases'
MODULE_DIR.mkdir(parents=True, exist_ok=True)
CASE_DIR.mkdir(parents=True, exist_ok=True)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

CASES = {'ISIC_0000013': {'label': 'Melanoma', 'selection': 'Selected Attention true-positive example'}, 'ISIC_0000003': {'label': 'Benign', 'selection': 'Selected Attention true-negative example'}}

EMBEDDED_FILES = {'ISIC_0000013.jpg': {'base64': '...', 'sha256': '8be9d9fbd3c55768d4c3a5cd4124196ae3f7488eb3a050681c21dbfbe28ab8a8'}, 'ISIC_0000013_Segmentation.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAA/4AAAL/CAAAAAD63EMAAAAPI0lEQVR4nO3d65LbNhaFUWgq7//Kmh92t3UhJYriBTh7raqpSVxO0gLOR0Lsi1sDAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAADnU5+wPgj6u94HBGbj/XxYt7ff4lG8P+TNlO/ha9aH0n6l/8z8J6RmwPdz2/XeKZ+m0Oe/vf2R9ARdcXf7f+3wNbk//2HqvVP52S/+Y+bfbV79c/e/L2cluTvc4s8p/PDLwt3BaxF7O1mWtrl+XP8H8+M7Do/m6X2IXB2sabim+WedV5/uU2Lf/6ArhjcrbwPunL0t/48h+f/0/bR1YwNhvY/wHd7DZd3/4OmOXJ/xDeX2B8joDPuWls4Ij05j57sPD3wQTT8r1jbryTO/XJ5xnhkVH53kHn7t+t8pUCbMSkfO+ot93Lvkro93fCOwblex0+dbOtLOHJ/9c6rL/Lj4n+uE18q9vSbC3vuPt/qdv6O/7I6IVbxDe6L8z28or5+EL39bdmh3nB4X+57X6KD3TBvWGh5++sG6R+O8wsw7HMTetffu/u4Wwxc8zGIg+tXwaq3xYzy2wsMU7rE2wxczz6W2Do+gf/6NmT/N8bvZ/RP352I/+3xq9n/FfAPuSfQP9Mkn8E/TPFY+F3CpVjs7lnIt4plH/79xOD7DvNGLxXK/87Nj+dCXincP7N/ofz6C9b7Ysbb8j/jep9VH99vPLf2R9A17RBae7+L6if2uQ/L6P+jFfJJPnH038u+aP/WPKflVNFzivlnvznJDWR9Fq5IX+a/lP5os85cUUYhTzu/jPi6g98xbjkT4ktwThkcfefEFt/8CvP5HL/KLwAA5HEbt8Lj7+ZiCgO/3fUbwmSyJ8H+s/hqHfH6N8yHNXZ4Vvqf2Q+SrO9t+Q/wYjUZW9vyX+aKSnKxt6S/yyDUpEn/zfUP296bazY2FzUbxjmJX5H5jr5qwzErt2Q/0I/f1LgxK8yElv2j/q/ZJhG470/xJI/xJI/m/HuaTTyh1jyh1jyh1jyh1jyZzue/Q1G/mxI/2ORP1vS/1Dk/8vkbsEqjsSXaf9hajdjpMbh7t9aU/+WrOU45N+aid2W1RyG/Nmc/kchf7an/0HIv5lWUsm/eVZNKvmzA+epMcifvbkWdOu/sz8Aarv++Z/3V12SP3u4Tv8scPri8N9M6R7+3vXpmkNZa+b0AAatR+7+EEv+EEv+HMHZv0vyh1jyb578kUr+6ieW/NVPLPmr/whWuUvy5xD675H8OYb+OxSe/9VUHsZK9yf5yzHM48GSh61PwXd/9R/NivcmN3+zSLzc/CGe/DmOE1dnYvM3iWew6n1JfRhrDk+TOnI9ir37cxZX3n6kXorN4KlSx643ofug/tOFTl5fQg//Zg9i8wfkz1m8/+pAaP5m73z24Hyh+dMB/Z9O/pxG/2fLzN/cQUvNH2jy50xOYSeTPyfS/7kyv/zN1HUpcxjPFLni6u9W5DyeJ3G51d+zxIk8TeB7f/XDH3n5q79v9udAcfmbrt7ZoeOEvdMyWmMIG8vTZN391Q83svIHbkTl7+Y/Cjt1jKT8zRTcScofuCN/iBWUv7M/3MvJX/0jsVuHyMmfoej/CPKHWDH5u5vAo5j8GYzr9QHkD7FS8ncvgScp+TMa3/N7APnTJ+e1A8ifTl2ba8DeUo5Y5mhYKSN6hpC7v/rHZe/2E5I/A9P/buRP9/S/l4z8zQ9MyMgfmCB/+uf0thP5MwD970P+EEv+EEv+jMDpfxfyZwj634P8IVZE/u4cMCUhf/VXYBd3EJC/uanBPm6vfv6mBmaUz1/9MKd6/uqvw15urnj+JqYSu7m14vlTiv43Jn8Gov9t1f4xqqaloNojeyx3fwZzdVHfjPwhlvwhlvwZjtP/VuQPseQPseQPseQPseQPseTPeDz630jp/E1JVXZ2G6Xz99Xh8Erp/IFX5M+InP43UTt/p394oXb++ocXiudPVU7/W5A/xCqev3sEzKudv/rhhdL5qx9eqftoXPvF1R3d45S9+6u/Ojv8var5m4367PHXquZPAP1/q+obKJMRo+oIH6Ho2qk/StEp3p/DP+NztV9J/hCraP5Og/Be0fzJ4vS/jvwhlvwhlvypwOl/FflTgv7XkD816H8F+VOE/j8nf6rQ/8fkTxn6/1TVL48zCaGqDvQ+qq6W/HNVnekdOPxDLPlDLPlTjTd+i1XN3/u/YPpfqmr+JNP/QvKnIP0vUzZ/p/9o+l+kbP76z6b/Jermr394o3D+wGvypyan/wXkT1H6f0/+EKtw/q7+8Frh/Ann+v+W/ClL/+8Uzt/n/ePp/43C+cPVFeClyrdIO09rrfaQf8fdn/LcB+bIH2JVzt+ZD16qnD/84fQ/Q/4E0P+02gdku86N2sO+RvEV0T93is/7p6ovh/55UH3kP1F9LeTPs+pTv1j1hZA/k6oP/jKe/BPJfaE1+UMw+UMs+ZPJ6b+Vz98ew7zi+Xu+C/OK5w9znAzlD8Gq5+/0D7Oq569/5jj9Z9Rhn5kUMf2vZCyA/pmREcCc8od/YI78IVZG/tknPF7Ifl+Ykb/+YUJI/sAz+UMs+UMs+UMs+UMs+UOskPyzP7sL00Ly93l/eBaSP/BM/hArJX+nf3iSkj/wRP4QKyZ/p394FJM/8Ej+EEv+EEv+EEv+EEv+ZIv+bjD5Qyz5Qyz5Ey759C9/iCV/iCV/iCV/iCV/0gU/+5M/xJI/xMrJ38/7gAc5+QMP5E+64HNhUP7Bu8wrnvxDrtz+5Q+x/csf2jX0CpD0hjhzh1ksKYY/ku7+ebsLLyXlD9yRP/yV9+4wKn+nf7gVlb/+4VZW/vqHG2H5A//IH2LJH2LJH2Kl5e/ZH/xKyx/4JX+IFZe/0z/8iKwh72u7WSSuhri7f2uBuwyTIvPXP7SWmr/+ocXmD+Tm7/bPk7yhSM2/XfL2Gh7E5p94rYd7wfnDncD7gfyhtRZZv/yhtZZZv/whl/whlvwhlvwhlvwhVnL+kc964Z/k/OFX5r1A/hArOv/MKz78iC/Az/2jxXYQffdvLXbfockfgskfYs+A8k/def6JnQH5Qyz5Qyz55578+Ct3Av47+wOAk1xauya338Jf/A9f+pPI6FuDH9d2cRGIYvKbRbil/yQmv3n0B8HkTyZnvSb/W46DhJE/xJI/mZz1mvzvmAiyyB9iyf+W238MW92aVXji80ERzH1rzd3/iblIYJf/kD951P+X/CGW/B94708O+T9wLqzPHv/w037Iov0b7v5EUf8t+T8yH5XZ3Tvyf2JC6rK39+RPDvU/kD/Ekv8z9whCyB9iyR9iyX+C0z8Z5D9F/0SQ/yT9k0D+0/RPAPnP0D/1yR9iyR9iyX+O0z/lyX+W/svxk9weyH+e/ilO/i/on9rk/4r+a7GfD+T/knmhMvlDLPkTw1nukRV54/dzRZeHv2csJn2Cu/8bl5n/Zyz2bYpVWeB6t0ru/wMy55Msy+f0PxxjPs3h/3NmaTR2bIb8VzBN1CB/iCX/Ndz+h2K75sh/lYuJogD5r6R/xif/tfQ/Cjs1S/6rmSpGJ3+IJX+Kc0qbJ//1zNUI7NIL8v+CyeqfPXpF/lSm/pfkD7HkD7HkT2HO/q/9d/YHADvR/lvu/hSl/vfk/w0T1i97s4D8KUn9S8h/OyauH/ZiEcv0nfs/BcTPAO6FuV7Ek//vXB7/FAAYh9Hdkrt/Fwz1UlZqU/o/nYn+gEd/lKL+T8ifStT/EflvyvSdy/p/Rv7Uof4PyR9iyR9iyZ8ynP0/JX+qUP/H5E8VvubqY/LfljsQA5E/xJI/ZTj9f0r+G3P6ZxymdQdXP/vjHKb5QxZsP/o/mGH+lMP/fkwjnZP/jvRP30zorpz/D2OSV7Boe3MFOIRBXsOqHcZ1YEfmeBXv/Q9jQvdjbdexboe6XpwBdmCKV3L3P9TFpNIRf8rPuS6eCXAed/9TXZqT6/es4FryP9pl5q9ZyyquZunOcG3tZumd/r9ihNezdqdT/xfM71cs3/n0v5Lh/ZYn/4xJ+xvw6I8hqX8L8j+fSf6cNduEZeyD9/8fMbbbsI5dcRVYxNRuxOG/K+aaI8kfYskfYjltdubvu//33wmY/L2CpnYjFrI/17+b8rLu6D9JxNRuxEJ27EXd2d8vZGo34r1/x+an/LLg99SV+Jr3If+ezc25+WcTvuWna1PP97TPVszS8OLe/ZvZzTj8Mxj1b0f+wwvLIezl7kv+47sognVMThExTwBM7IYsZhUR/RvXbVnPMor3b1J34L1/GbX7qP3qziL/OioXUvm1nUj+hdRtpO4rO5f8K6laSdXXdTr5l1Kzk5qvqgfyp3fq34386Zz69yP/Wuq1Uu8VdcTi1lPr639M6I4sbnHDXwtM6I4c/otTD/PkX93g/Q/+4XdO/uUJiDnyh1h+0i8dc3LZl/UNMOjTf7O5O4d/OqX+/VnjBOPd/s3lISxzhqEuAIbyKFY6RudXAJN4Au/9Y/TdV98fXVVWPUq3JwBzeAp3/yi9Vtbrx1Wd/LP02VmfH1UA+YfpsbQeP6YMVj7O5Pv/y+7PBS5P//4//00TeCKLn2ei88vMr2/n+fpi9M5nDwLNdbhj/5en/4LJ64BNyPTb4WXyV7f2fIUxeD2wC9zYqf+bKbs+/QrnsQ3c2Cf/hyG7Grpe+MQfe7jM/PXU33MaW8GtjW7//z6TYMB6Zne48/Pe/O7/P/2qAFM1CBvFo5/35tfVnxUwVYOwUSz0Qf+mahA2iuUWXgEM1SjsFB9ZcgUwVKPwiT8+Iu1K5M9n3vfvCjEMf8oPm9L+SOwWn5p/+2+aBmPDWGH6CmCYRmPHWO3uImCSBuTRH6u9+r4eRiB/1rtM/BUDsW18x0/rBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAov0fDTl2lT7dDXIAAAAASUVORK5CYII=', 'sha256': '5c006348e5d5258a659b9dabb8e8dea129cae333e72d781ee65ffd294c92ddb4'}, 'ISIC_0000003.jpg': {'base64': '...', 'sha256': 'e3092bd47d1955c117a18acb1e236222cae99acfbd393d0c7f07b4e6b8978857'}, 'ISIC_0000003_Segmentation.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAA/4AAAL/CAAAAAD63EMAAAARmklEQVR4nO3d23bbOBZFUapH/v+X1Q8VO75QEkGRIHD2nA9dSY2kWgHOIqGLnWUBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAICdblc/ACq6m6wp2CSOcv/5LwzX6OwQh/jV/rKYruH97+oHQAmr9T/4twzD9ZlGa8/rH3ZuvoZme2jx2flt9d/+Yr6G5vBPg/vKj5zx5yV/tmt/hu/KMDT5s5nX96qRP6/cf/zz68/ud/VPzEszPPW37tv6p3pet2/ARmZ3eObte7sBG5nDP0+ov7Y/Vz8AhuQZfQT584v2Uzj885P6Y8ifH9SfQ/58d2z9riVDkz/fHN2r/kfmjZkc95ebfUqrRmxc9ibEv0/vvfw1RzNjw7I1V7svyynb8PW7cvzoevX/7dRTujEbk3251JfmDt2Jr9+VY9t34Tv5Obo5G5JtudKL5po3Z+DX2QzaiOzKhTbU2rA/A7e/LCZtSDblKk21XvOa/aFM2oBsyiV21Ppkp8ZvfzFpQ/Kxnyvs6fXx75mi/kkeZRj5X2BfCY9+1yxdzfI4kziS9fZWBY/eyZ+FcRuL/ejszW5vB/w3rmTehmI7+pq43OMYulHYia7U/5e5G4Jt6Ef7Xxi8EXjlvxv1f2U1RuAi3It5/8XwXc0OdKL+VebvUpa/B+0/YQSvY+1P8+/7eKj/KTN4GUt/DsU3MYbX8Mr/KdTfxnpdw2X3eGZ5B4N4BXf/w6l/D6t2BRfdY5ni3Yxif+7+h1L/ftauP5fcAxng9xjG3tz9j6P+N1nA3uQPseR/GPcuZiP/o6j/fdawM/kfxOQewSr2Jf9jmNtjWMeu5H8IU3sUK9mTt1qPYGYPZSh7cfc/gPqZk/zfp34mJX+G43rai/whlvzf5l7FrOT/LvUzLflDLPkzHieqTuT/JpN6Bqvah/zfY06ZmPzfon5mJv93qP8sVrYL+b/BjJ7H2vYg//1M6Jmsbgfy3818nsv6nk/+e5lOpif/ndTP/OS/j/rPZ41PJ/9dTCYVyB9iyZ9hOWOdTf57mEtKkD/Ekj/Ekv8Ozv7UIP8d/C001CB/iCV/iCV/iCV/xuU11pPJfwdT2ct9uVvsE3kRewcT2ZkpPYm7P8SSfzs3/96s+EnkzwT0fw75MwP9n0L+zUwiVcgfYsmfKThznUH+rcwhZcgfYskfYsm/kbM/dci/jfqvYuVPIP8mZvA61v548m9hAq9k9Q8nf6ah/6P5SuoW5u9yBvZIVrOB+odgZg/j8M9sXIUPI3+mo/+jyL+BUye1/Ln6AcBed1fkN1m+Fk6dg7h92QsjvJ/DfwP1j+L+ZS/syn7yZ0aaP4T8IZb8mZ2TwG7yZ3r630v+DbzGPCj97yR/CtD/Pm5oLUzZwD5H+f7j5zxkjVrIf2y3H1tkul/woV/qcHlu5Ll/C3eTSd3vrgxrDHQTQzSX27L8/uqAu6H/YCWayH8yv14N8LrgV5ahjf7LMPqe+xPLldwlsJWZKSV9/N39m6i/lvT9TL/8NUofl4qSE3D3J1zyJV3+LZInpa7gXZV/i+RzYmG5/csfYvuXP8SSfxOnfyqRP8Se/uUPseQPseQPsad/+cOS2r/8YVmWJfLbgckfYskf/nP/8r8hfI6lSdRs5In7ToDu/vAh7uouf4gl/xZxd4dQMfss/xYxzwnTpfQvf/gtpH/5w4qM/h1n22RMBcuyJMTh7t+m/kQQRP6N9E8d8ocH6j/Tkz/Ekn8rp3/KkH8z/ccof/qXPzxUvX+3snbVZ4JvKifi7g9PVb7ayx9iyR9iyR+eK3z6lz+8ULd/+cMrZfuXP7xUtX/5w2tF+5c/bFCzf/nDFiX7lz9sUrF/+cM2BfuXP8SSP8SSP2xU7/Qvf4gl/2b17gGkkn8r9ecqt/fyh82q9S9/2O5e6wIg/0a1tp9s8m9U+fu+kkb+0KLU8U/+EEv+0KLUsz/5Qyz5Qyz5Qwsv/SUrtfvsUGkC5A+x5A+x5N+m0smPfQrNgPzblHrXl3Tyh1jyh0Z1Tv/yb+T0Tx3yb6V/ypB/M/3HK3P6lz/Ekj/Ekn+zMic/dqsyA/JvVWXnQf6t1E8d8odY8odY8m/j7E8h8odY8odY8m/i7E8l8odY8odY8m/h7E8p8odY8od2Rc6B8odY8odY8m9Q5MTHAWrMgvy3q7Hj8En+EEv+EEv+mzn7U438IZb8IZb8t3L2pxz5Qyz5Qyz5Qyz5Qyz5Qyz5wx4l3gmSP+xSoX/5Qyz5b1ThWg/fyX8b9fNTgZmQP+w0f//yh73us18A5L/J7NsMa+S/hfopSf6w3+Q3BvlDLPlvMPklHh6QP8SSP+x3u/oBvEf+EEv+EEv+G0x+woMH5A+x5A/7Tf6esPzhDXP3L394x9T9yx/eMnP/8odY8t/CO3+UJH94z8Snf/lDLPlv4vRPRfKH90x8b5A/xJL/JhO/ugMPyX8L9VOS/DdQPzXJH2LJH2LJ/zVnf56Y+H0/+UMu+UMs+cM7Zj77yx9yyR9iyR/eMPXZX/6ved+Ph+auX/4vqZ+y5P+C+qlL/s+pnycmP/vL/zn1U5n8n1E/pcn/CfXz1Oxnf/lDLvlDLPk/5uzPU9Of/eUPueQPseT/kLM/T81/9pc/5JI/xJL/I87+PFXg7C9/yCV/iCX/B5z9earC2V/+D6ifAPJfpX4SyB9iyR9iyX+Nsz8R5A+x5A+x5A+x5A97lHh9SP6wS4X+5Q/7FOhf/rDT/P3LH2LJf8X8V3XYQv6w1/T3CflDLPlDLPlDLPmvKPF9nOAl+cNe098n5A+x5A+x5A+x5A+x5A+x5A+x5A+x5L9i+q/kgE3kv2L6T3PAJvKHWPKHWPJf4bk/GeQPseQPseS/wiv/bDL9s0T5Qyz5Qyz5r3H6Z5PZT//yX6V/EsgfYskfdpv9lCh/iCV/iCX/VbO/oksXs5/95Q+55A+x5L/G2Z8Npj/7yx92mr9++UMu+UMs+UMs+a/wyh8Z5A+x5A+x5P+bsz8h5A+x5A/7FDglyh9iyR/28aFfYF7yh1gFDjBHK/CKDucrUU6JP8SBtM8mNcJx+P9G/WxSo375Qy75Qyz5f+XszyZFzv7yh2ZV6pc/5JI/xJI/NCpz9pf/V175Y4M69cv/C/UT5s/VDwCmUuje7+4PTUrVL/9/nP15qVb98v+kfl4qVr/8P6ifl6rVL3/IJX+IJf//OPvzUrmzv/z/qrezHK3gjMgfNilYv/xhk4r1y/9Dyd2Fp+QPseQPW5R8b0j+f5XcXXhK/hBL/hBL/hBL/rBFyXeG5f9Xyd2Fp+QPseQPG9Q8Hcr/Q8395RhFp0P+n4ruMAeoOhtV/1w7+ewfK8pW4u7/Tdl95g11p8Lf8gPP1G1/cfeHp0rXL394onb98odc8odY8oeHip/95f+d9/35onr98odc8odY8ocHyp/95f9d/Q1ns4BhkD/Ekj/Ekv93AQc+tkkYBfn/cEvYdViWRf4r9E8K+f+mf0KmQP4rInaepzJmQP5rMvaex0ImQP6rQnafcPKHWPKHX1JOf/Jfl7L/rInZffk/EDMB/JKz9/J/JGcG+C5o5+X/UNAUEEr+EEv+8E3SqU/+jyXNAR+idl3+T0RNAsuypO25/J/JmgXidlz+T4VNA2HkD7HkD5/STnvyhw9p9csfPsTVL3/4K69++UMu+UMs+cOyLJFnf/nDsiyZ9csfliW0/tA/9Wb3qx8APcRW4O7/VOxcRMndZfmTLrd++ZMuuH75v5A8Gxmid1j+L0RPR4Ds/ZX/K9nzUV347sr/pfAJKS19b+VPrvT65U+u+Prlv4Epqcm+yn8Lc1KRXZU/qdS/yJ9Q6l8W+W/i6/7KUf+yLPInkvr/I3/yqP8v+W9gWqhJ/hBL/sRxmvsg/y3MSyV285P8NzExddjLf/5c/QCgI+1/4+5PEPV/J39yqP8H+UMs18ONfO5/emb9F3d/Qqj/N/mTQf0r5L+R6Zmb/Vsj/63Mz8zs3ir5b2aC5mXv1vnUH9Vp/yF3f4pT/2Py384czciuPSH/BiaJWuTfQv+UIv8m+p+NHXtG/m1ME4XIv5H+qUP+VOZq/ZT8KUz9z8m/lYmah716Qf6Upf5XfOb/XTffCWhQ6n/J3b/Z7dfPzBlzkn+724Mfw2Tkv8Nt5UeMxt68Jv89bg/+CVMxuHvdvy2dV/+GY7Rfc/ffy3SNzf5sIH+IJX+IJX9KcvbfQv5UpP5N5E9B6t9G/tSj/o3kTznq30r+VKP+zeR/DCPHhOQPseRPMQ5i28n/IIZuEDaigcU6kC/7G4CJbmCxDuYScDET3cDh/2Cmj3nI/2j6ZxryP5z+mYX8j6d/JiH/E+j/Ota+hfzPYAaZgvxPoX9mIP9z6P8a1r2J/E9iDhmf/M+if4Yn/9Povz9r3sZ6ncpXAPRlnNtYr7O5AnRknNtYrw5cAbowy8089+/AXPZgldtZsy7c/09mjndx9+/CdJ7L+u4jf+an/p3kD7Hk34Xn/oxI/kzP2X8v+UMs+ffg7M+Q5M/snP13kz+TU/9+8mdu6n+D/CGW/CGW/Jmas/875N+B9/1Oo/63yJ+Jqf898mde6n+T/Dswpeewru+SP7NS/9v+XP0AYA/tH8EqduG1/0OZ2oM4/HdhXo9kNY8if2aj/sPIn8mo/zjyZy7qP5D8IZb8IZb8mYqz/5Hkz0zUfyj5Qyz59+GuxYDkz0RcRY8lf+ah/oPJvxOT+z5reDQr2pGv+3uDST2Bu39HJng/a3cG3+6D8Wn/JO7+DE/9Z5F/T+Z4D6t2Gvl3ZZLbWbPzyJ+xqf9E8u/LMDeyYGeyut1597+B+TyVu393Jno7a3Uu63uJ+7LcnAJeMp0ns8DX0f8rpvNkDv8QS/4My83/bPJnVOo/nfwZlPrPJ3/GpP4O5A+x5H8d9zcuJn+G5NrYg/wZkfq7kP+FzDjXkv+V9M+l5H8p/a+zLn3I/1rmnAvJ/2L65zryv5r+f7Mmncj/cmb9JyvSi5Uegu/88YWZ7MbdfwgmniuYu2E4AfxlJrtx9x+Gqac3+UMs+UMs+Y/D6Z/O5M9gXAX7kf+oVMDp5D+Q27cf65+zmbHB/H33/7akfhDARHb05+oHwHe3ZbkrgD4c/sejfjqRP0Nx7evJag8s7sm/aezM3Z9hqL83+TMK9Xcn/4Fl9ZD1px2D/EeWVETSn3UY8h9aThM5f9KRyJ8RqP8S8mcA6r+GdR/cv68BKPspADN4GUs/vo+vASjavxG8jrWfR8X8zd+lPPefR8FUCv6RpmL953L/b8eKHARM38VswKwKXAEM39XswMTun/s34bXA5A3AJtQwXf8GbwRe+qthtppme7xFyb+IuXqa69HWJf8qZipqpsdamvzLmKepeR5pdb7RNz3cvv0NBgzCZtQx8Kv/H+9PGrex2I9CBu3fjA3Lc/9CxuxszEfFstibcu7Lcvs8BYzw9QEmbGA2p6Lvr7Jd2r8BG5ndKerrq2wX9m++hmZ7ElzWv/Eam/0J8e01gV5M1+BsUJLO/Ruu0XnjL0nfHtU/PPlH6Vmk+sdnj+J0egZgsibg7h+nT5fqn4H88/QoU/1TkH+g89tU/xzsU6RTn/+bqWm4+0c6s1D1z8NehWq8/2//2kETNRF3/1Btld52/B7GJ39eu/34J0XIn5duKz96/WsZn/xTbe/09uDH7/1XGYBv9E2T29NXANU/F/nT6vG7AOqfjMN/rPdaXfvd6p+N/Nnnd+vqn44ty7Xxkz8PR+S+6VcxMLsWbFv/j0fk/vqXMDQbl+z9T/HejdDM7F24l1cAE1KYl/7CqTuZ/NPpP5j84z3v39WhMrvL8uTvADIfpdlePqz0bzxqc/jng8/xxZE/n9Sexo7zjU/yJnH355uW7+0BlHNfuv9d4AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAJDj/0DxhDgJPRfGAAAAAElFTkSuQmCC', 'sha256': 'f1c62aee591c8cbef5a3c1b6b5033f84b630c5649927389d17b53351fc5da521'}}

if IMAGE_ID not in CASES:
    raise ValueError('Choose one of the two embedded image IDs using the selector.')
for name, record in EMBEDDED_FILES.items():
    content = base64.b64decode(record['base64'])
    if hashlib.sha256(content).hexdigest() != record['sha256']:
        raise ValueError('Embedded reference file integrity check failed: ' + name)
    (CASE_DIR / name).write_bytes(content)

MODULES = {'attention_pipeline_core.py': '"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', 'bestcase_comparison_helpers.py': 'from pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nfrom attention_pipeline_core import MelanomaPipeline\nIMG_SIZE = 256\nCLS_IMG_SIZE = 224\nMASK_MODE = \'mask_crop\'\nSEG_IMAGE_SIZE = (256, 256)\nSEG_THRESHOLD = 0.5\n\ndef get_lesion_region(image, unet_model, pad_frac=0.15, mode=MASK_MODE):\n    """Runs the trained U-Net on a full-res image to get a lesion mask, then applies\n    pixel-masking and/or cropping (per `mode`) before resizing for the classifier."""\n    h, w = (image.shape[0], image.shape[1])\n    seg_input = tf.image.resize(image, [IMG_SIZE, IMG_SIZE]) / 255.0\n    seg_input = tf.expand_dims(seg_input, 0)\n    pred_mask_small = unet_model(seg_input, training=False)[0, ..., 0].numpy()\n    pred_mask_small = (pred_mask_small > 0.5).astype(np.uint8)\n    pred_mask_full = tf.image.resize(pred_mask_small[..., np.newaxis], [h, w], method=\'nearest\').numpy()[..., 0].astype(np.uint8)\n    ys, xs = np.where(pred_mask_small > 0)\n    lesion_found = len(xs) > 0 and len(ys) > 0\n    work_image = image.numpy() if hasattr(image, \'numpy\') else np.array(image)\n    if mode in (\'mask\', \'mask_crop\'):\n        if lesion_found:\n            work_image = work_image * pred_mask_full[..., np.newaxis]\n    if mode in (\'crop\', \'mask_crop\') and lesion_found:\n        x_min, x_max = (xs.min() / IMG_SIZE, xs.max() / IMG_SIZE)\n        y_min, y_max = (ys.min() / IMG_SIZE, ys.max() / IMG_SIZE)\n        pad_x = (x_max - x_min) * pad_frac\n        pad_y = (y_max - y_min) * pad_frac\n        x_min = max(0.0, x_min - pad_x)\n        x_max = min(1.0, x_max + pad_x)\n        y_min = max(0.0, y_min - pad_y)\n        y_max = min(1.0, y_max + pad_y)\n        x1, x2 = (int(x_min * w), int(x_max * w))\n        y1, y2 = (int(y_min * h), int(y_max * h))\n        work_image = work_image[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1), :]\n    out = tf.image.resize(work_image, [CLS_IMG_SIZE, CLS_IMG_SIZE])\n    return out.numpy()\n\ndef predict_mask_for_pil(model, image_pil):\n    resized = image_pil.resize((SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)\n    x = np.asarray(resized, dtype=np.float32) / 255.0\n    probability = model.predict(x[None, ...], verbose=0)[0, ..., 0]\n    binary = (probability >= SEG_THRESHOLD).astype(np.uint8)\n    return (probability, binary)\n\ndef extract_roi_from_binary_mask(image_pil, binary_mask, padding_fraction=0.1):\n    image_array = np.asarray(image_pil.convert(\'RGB\'))\n    h, w = image_array.shape[:2]\n    mask = np.asarray(Image.fromarray((binary_mask * 255).astype(np.uint8)).resize((w, h), Image.Resampling.NEAREST)) > 0\n    ys, xs = np.where(mask)\n    if len(xs) == 0 or len(ys) == 0:\n        return (image_pil.copy(), (0, 0, w, h), True)\n    x_min, x_max = (int(xs.min()), int(xs.max()))\n    y_min, y_max = (int(ys.min()), int(ys.max()))\n    box_w, box_h = (max(1, x_max - x_min + 1), max(1, y_max - y_min + 1))\n    pad_x = int(round(box_w * padding_fraction))\n    pad_y = int(round(box_h * padding_fraction))\n    x0, y0 = (max(0, x_min - pad_x), max(0, y_min - pad_y))\n    x1, y1 = (min(w, x_max + pad_x + 1), min(h, y_max + pad_y + 1))\n    if x1 - x0 < 8 or y1 - y0 < 8:\n        return (image_pil.copy(), (0, 0, w, h), True)\n    return (image_pil.crop((x0, y0, x1, y1)), (x0, y0, x1, y1), False)\n\ndef hard_overlap(truth, prediction):\n    truth = np.asarray(truth, dtype=bool)\n    prediction = np.asarray(prediction, dtype=bool)\n    if truth.shape != prediction.shape:\n        raise ValueError(f\'Mask shapes differ: {truth.shape} and {prediction.shape}\')\n    intersection = int(np.logical_and(truth, prediction).sum())\n    total = int(truth.sum()) + int(prediction.sum())\n    union = int(np.logical_or(truth, prediction).sum())\n    return {\'dice\': 2.0 * intersection / total if total else 1.0, \'iou\': intersection / union if union else 1.0}\n\ndef load_pair(segmenter_path, classifier_path):\n    seg = tf.keras.models.load_model(str(segmenter_path), compile=False)\n    cls = tf.keras.models.load_model(str(classifier_path), compile=False)\n    if tuple(seg.input_shape[1:]) != (256, 256, 3):\n        raise ValueError(\'Expected a 256 x 256 RGB segmenter.\')\n    if len(cls.inputs) != 1 or tuple(cls.input_shape[1:]) != (224, 224, 3):\n        raise ValueError(\'Expected a single-input 224 x 224 RGB classifier.\')\n    return (seg, cls)\n\ndef check_score(score):\n    score = float(score)\n    if not np.isfinite(score) or not 0 <= score <= 1:\n        raise ValueError(f\'Invalid sigmoid model score: {score}\')\n    return score\n', 'full_stage_visuals.py': '"""Export real inference stages from four existing melanoma pipelines. No training."""\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport zipfile\nfrom datetime import datetime, timezone\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nimport matplotlib.pyplot as plt\nfrom scipy import ndimage\n\nfrom attention_pipeline_core import MelanomaPipeline, image_tensor, roi_tensor\nfrom bestcase_comparison_helpers import (\n    load_pair, get_lesion_region, predict_mask_for_pil,\n    extract_roi_from_binary_mask, hard_overlap, check_score,\n)\n\nORDER = [\'Attention U-Net\', \'U-Net (uploaded notebook)\', \'ResUNet\', \'TransUNet\']\nSLUGS = dict(zip(ORDER, [\'attention_unet\', \'unet\', \'resunet\', \'transunet\']))\nTITLES = [\'Original image\', \'Ground-truth mask\', \'Segmentation probability\',\n          \'Predicted lesion mask\', \'Mask overlay\', \'Isolated lesion\',\n          \'Classifier lesion ROI\', \'Grad-CAM heatmap\', \'Grad-CAM on ROI\']\nKEYS = [\'original\', \'ground_truth\', \'probability\', \'mask\', \'overlay\',\n        \'isolated\', \'roi\', \'heatmap\', \'gradcam\']\n\n\ndef array_rgb(pixels):\n    return np.uint8(np.clip(np.asarray(pixels), 0, 255))\n\n\ndef signed_logit_gradcam(model, image_batch, target_class=1):\n    """Follow each saved classifier\'s augmentation, backbone and head exactly.\n\n    All panels explain the melanoma logit, giving the same target across models.\n    Explicit input watching also supports fully frozen EfficientNet backbones.\n    """\n    candidates = [layer for layer in model.layers\n                  if isinstance(layer, tf.keras.Model)\n                  and \'efficientnet\' in layer.name.lower()\n                  and len(layer.output_shape) == 4]\n    if len(candidates) != 1:\n        raise ValueError(f\'{model.name}: expected one nested spatial backbone; \'\n                         f\'found {[layer.name for layer in candidates]}\')\n    backbone = candidates[0]\n    index = model.layers.index(backbone)\n    final = model.layers[-1]\n    if not isinstance(final, tf.keras.layers.Dense) or final.units != 1:\n        raise ValueError(\'Expected the saved one-unit sigmoid classification head.\')\n    batch = tf.convert_to_tensor(image_batch, dtype=tf.float32)\n    with tf.GradientTape() as tape:\n        tape.watch(batch)\n        value = batch\n        for layer in model.layers[:index]:\n            if not isinstance(layer, tf.keras.layers.InputLayer):\n                value = layer(value, training=False)\n        feature = backbone(value, training=False)\n        tape.watch(feature)\n        value = feature\n        for layer in model.layers[index + 1:-1]:\n            value = layer(value, training=False)\n        logit = tf.linalg.matmul(value, final.kernel)\n        if final.use_bias:\n            logit = logit + final.bias\n        score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        probability = tf.math.sigmoid(logit)[0, 0]\n    gradient = tape.gradient(score, feature)\n    if gradient is None:\n        raise RuntimeError(f\'{model.name}: Grad-CAM gradient could not be traced.\')\n    pooled = tf.reduce_mean(gradient, axis=(1, 2), keepdims=True)\n    heat = tf.nn.relu(tf.reduce_sum(feature * pooled, axis=-1)[0])\n    heat = heat / (tf.reduce_max(heat) + 1e-8)\n    return heat.numpy(), float(probability.numpy())\n\n\ndef detailed_attention(image_path, spec):\n    pipeline = MelanomaPipeline.load(spec[\'run_dir\'])\n    result = pipeline.predict_file(image_path, explain=False)\n    full = image_tensor(image_path, (224, 224))[None, ...]\n    roi = roi_tensor(result[\'roi\'])[None, ...]\n    heat = pipeline.gradcam(full, roi, target_class=1)\n    return dict(probability=result[\'probability_map\'],\n                mask=np.asarray(result[\'mask\']) > 0,\n                roi_pixels=roi.numpy()[0], heatmap=heat,\n                score=check_score(result[\'score\']),\n                threshold=float(pipeline.configuration[\'threshold\']),\n                mask_threshold=float(pipeline.configuration[\'segmentation_threshold\']),\n                mask_method=pipeline.configuration[\'segmentation_method\'],\n                classifier_method=pipeline.configuration[\'method\'],\n                roi_note=result[\'roi_reason\'],\n                full_classifier_input=full.numpy()[0])\n\n\ndef detailed_peer(name, image_path, spec):\n    segmenter, classifier = load_pair(spec[\'segmenter\'], spec[\'classifier\'])\n    with Image.open(image_path) as handle:\n        original = handle.convert(\'RGB\')\n    if name.startswith(\'U-Net (\'):\n        rgb = tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(image_path)), channels=3),\n                      tf.float32)\n        seg_input = tf.image.resize(rgb, (256, 256)) / 255.0\n        probability = segmenter(seg_input[None, ...], training=False)[0, ..., 0].numpy()\n        mask = probability > 0.5\n        roi_pixels = get_lesion_region(rgb, segmenter, pad_frac=0.15, mode=\'mask_crop\')\n        note = \'Background masking and 15% bounding-box crop\'\n    elif name == \'ResUNet\':\n        probability, mask = predict_mask_for_pil(segmenter, original)\n        roi, _, fallback = extract_roi_from_binary_mask(original, mask, padding_fraction=0.10)\n        buffer = BytesIO()\n        roi.save(buffer, format=\'JPEG\', quality=95)\n        pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n        roi_pixels = tf.cast(tf.image.resize(pixels, (224, 224), method=\'bilinear\'),\n                             tf.float32).numpy()\n        note = \'Full-image fallback\' if fallback else \'10% RGB bounding-box crop, quality-95 JPEG\'\n    elif name == \'TransUNet\':\n        pixels = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n        if pixels is None:\n            raise ValueError(\'OpenCV could not decode the original image.\')\n        pixels = cv2.cvtColor(pixels, cv2.COLOR_BGR2RGB)\n        seg_pixels = cv2.resize(pixels, (256, 256)).astype(np.float32) / 255.0\n        probability = segmenter.predict(seg_pixels[None, ...], verbose=0)[0, ..., 0]\n        mask = probability >= 0.5\n        masked = seg_pixels * mask[..., None].astype(np.float32)\n        roi_pixels = tf.image.resize(masked, (224, 224)).numpy() * 255.0\n        note = \'Background-masked 256-pixel full frame resized to 224\'\n    else:\n        raise ValueError(name)\n    batch = np.asarray(roi_pixels, dtype=np.float32)[None, ...]\n    score = check_score(classifier.predict(batch, verbose=0).ravel()[0])\n    heat, reconstructed_score = signed_logit_gradcam(classifier, batch, target_class=1)\n    if not np.isclose(score, reconstructed_score, atol=2e-5, rtol=2e-5):\n        raise RuntimeError(f\'{name}: Grad-CAM head does not reproduce the saved \'\n                           f\'classifier score ({reconstructed_score} vs {score}).\')\n    return dict(probability=probability, mask=np.asarray(mask, dtype=bool),\n                roi_pixels=roi_pixels, heatmap=heat, score=score, threshold=0.5,\n                mask_threshold=0.5, mask_method=\'single\', classifier_method=\'single\',\n                roi_note=note, full_classifier_input=None)\n\n\ndef stage_panels(original, truth, result):\n    rgb = np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR))\n    mask = np.asarray(result[\'mask\'], dtype=bool)\n    if mask.shape != (256, 256) or np.shape(result[\'probability\']) != (256, 256):\n        raise ValueError(\'Expected a 256 x 256 segmentation map.\')\n    overlay = rgb.astype(np.float32).copy()\n    overlay[mask] = .65 * overlay[mask] + .35 * np.array([35, 195, 145])\n    border = mask & ~ndimage.binary_erosion(mask)\n    overlay[border] = [255, 215, 30]\n    roi = array_rgb(result[\'roi_pixels\'])\n    raw_heat = np.asarray(result[\'heatmap\'], dtype=np.float32)\n    if not np.isfinite(raw_heat).all() or raw_heat.ndim != 2:\n        raise ValueError(\'Grad-CAM must be a finite spatial map.\')\n    heat = tf.image.resize(raw_heat[None, ..., None], roi.shape[:2],\n                           method=\'bilinear\')[0, ..., 0].numpy()\n    heat = np.clip(heat, 0, 1)\n    colors = plt.colormaps[\'jet\'](heat)[..., :3]\n    cam = .62 * roi.astype(np.float32) + .38 * colors * 255\n    # Do not paint an apparent focus when every positive contribution is zero.\n    if float(raw_heat.max()) <= 1e-8:\n        cam = roi.copy()\n    return dict(original=rgb, ground_truth=np.uint8(truth) * 255,\n                probability=np.asarray(result[\'probability\'], dtype=np.float32),\n                mask=np.uint8(mask) * 255, overlay=array_rgb(overlay),\n                isolated=rgb * mask[..., None], roi=roi,\n                heatmap=heat, gradcam=array_rgb(cam))\n\n\ndef plot_grid(results, output, filename, keys=KEYS):\n    columns = len(keys)\n    width = 3.0 * columns + 2.7\n    fig, axes = plt.subplots(len(results), columns, figsize=(width, 2.9 * len(results) + 1.9),\n                             squeeze=False, facecolor=\'white\')\n    for row, result in enumerate(results):\n        for col, key in enumerate(keys):\n            ax = axes[row, col]\n            if key in (\'probability\', \'heatmap\'):\n                ax.imshow(result[\'panels\'][key], cmap=\'viridis\' if key == \'probability\'\n                          else \'jet\', vmin=0, vmax=1)\n            elif key in (\'ground_truth\', \'mask\'):\n                ax.imshow(result[\'panels\'][key], cmap=\'gray\', vmin=0, vmax=255)\n            else:\n                ax.imshow(result[\'panels\'][key])\n            ax.axis(\'off\')\n            if row == 0:\n                title = TITLES[KEYS.index(key)]\n                ax.set_title(title, fontsize=13, pad=14, fontweight=\'bold\')\n            if key in (\'heatmap\', \'gradcam\') and result[\'no_positive_cam\']:\n                ax.text(.5, -.035, \'No positive activation\', ha=\'center\', va=\'top\',\n                        transform=ax.transAxes, fontsize=10)\n        name = \'U-Net*\' if result[\'name\'].startswith(\'U-Net (\') else result[\'name\']\n        label = (f\'{name}\\nDice {result["dice"]:.4f}\\nIoU {result["iou"]:.4f}\\n\\n\'\n                 f\'Prediction: {result["prediction"]}\\nScore {result["score"]:.4f}\\n\'\n                 f\'Cutoff {result["threshold"]:.4f}\')\n        axes[row, 0].text(-.075, .5, label, transform=axes[row, 0].transAxes,\n                         ha=\'right\', va=\'center\', fontsize=11, linespacing=1.4)\n    image_id = results[0][\'image_id\']\n    true_label = results[0][\'true_label\']\n    fig.suptitle(f\'{image_id} | Actual label: {true_label} | All pipeline stages\',\n                 fontsize=22, fontweight=\'bold\', y=.985)\n    fig.text(.52, .035, \'Same selected image in every row. Grad-CAM target: melanoma class \'\n             \'(Attention: ROI branch). Maps are individually normalised; colours are not comparable in magnitude.\',\n             ha=\'center\', fontsize=11)\n    fig.text(.52, .012, results[0][\'selection_note\'] + \'; different training splits and ROI methods. \'\n             \'*The uploaded U-Net++ notebook implements U-Net. This is not an overall model ranking.\',\n             ha=\'center\', fontsize=11)\n    fig.subplots_adjust(left=2.5 / width, right=.992, top=.91, bottom=.08,\n                         hspace=.25, wspace=.08)\n    fig.savefig(output / filename, dpi=220, bbox_inches=\'tight\', facecolor=\'white\')\n    plt.close(fig)\n\n\ndef resolve_saved_models(drive_root, comparison_root, preferred_comparison=None):\n    """Resolve the checkpoints belonging to the completed comparison, without fitting."""\n    drive_root, comparison_root = Path(drive_root), Path(comparison_root)\n    preferred = Path(preferred_comparison) if preferred_comparison else None\n    manifests = ([preferred / \'comparison_provenance.json\'] if preferred else [])\n    manifests += sorted((comparison_root / \'comparisons\').glob(\'*/comparison_provenance.json\'),\n                        key=lambda p: p.parent.name, reverse=True)\n    manifests = [p for p in manifests if p.is_file()]\n    if not manifests:\n        raise FileNotFoundError(\'The completed comparison folder was not found. \'\n                                \'Use the same Drive account as your completed run.\')\n    manifest_path = manifests[0]\n    manifest = json.loads(manifest_path.read_text())\n    models = {\'Attention U-Net\': {\'run_dir\': Path(manifest[\'attention_saved_run\'])}}\n    originals = {\n        \'U-Net (uploaded notebook)\': (\n            drive_root / \'Melanoma_Comparison_Models/unet_segmenter.keras\',\n            drive_root / \'Melanoma_Comparison_Models/unet_classifier.keras\'),\n        \'ResUNet\': (\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/resunet_best.keras\',\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/efficientnet_b0_final.keras\'),\n        \'TransUNet\': (drive_root / \'43022/TransUNet_ISIC2016.keras\',\n                     drive_root / \'43022/EfficientNetB0_Augmented_Final.keras\'),\n    }\n    records = {row[\'Pipeline\']: row for row in manifest[\'peer_model_provenance\']}\n    missing = []\n    for name in ORDER[1:]:\n        if name not in records:\n            raise ValueError(\'Comparison provenance is missing \' + name)\n        record = records[name]\n        folder = comparison_root / \'trained_models\' / record[\'Code hash\'] / name.split()[0].lower()\n        cached = (folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\')\n        if \'NEW\' in record[\'Weights\']:\n            pair = cached  # Do not silently substitute different weights.\n        else:\n            pair = cached if all(p.is_file() for p in cached) else originals[name]\n        missing.extend(str(p) for p in pair if not p.is_file())\n        models[name] = dict(segmenter=pair[0], classifier=pair[1])\n    attention = models[\'Attention U-Net\'][\'run_dir\']\n    missing.extend(str(attention / p) for p in (\n        \'models/attention_unet_best.keras\', \'models/selected_final_classifier.keras\',\n        \'tables/selected_configuration.json\') if not (attention / p).is_file())\n    if missing:\n        raise FileNotFoundError(\'Saved checkpoints are missing. No training was started.\\n\'\n                                + \'\\n\'.join(missing))\n    return models, manifest_path.parent, manifest\n\n\ndef export_all_stages(image_path, mask_path, models, destination, previous_comparison=None,\n                      source_manifest=None, true_label=None, selection_note=\'Selected illustrative case\'):\n    if true_label not in (\'Benign\', \'Melanoma\'):\n        raise ValueError(\'The verified actual label must be supplied explicitly.\')\n    image_id = Path(image_path).stem\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with Image.open(image_path) as image:\n        original = image.convert(\'RGB\').copy()\n    with Image.open(mask_path) as mask:\n        truth = np.asarray(mask.convert(\'L\').resize((256, 256), Image.Resampling.NEAREST)) > 127\n    previous = None\n    if previous_comparison and (Path(previous_comparison) / \'single_image_comparison.csv\').is_file():\n        previous = pd.read_csv(Path(previous_comparison) / \'single_image_comparison.csv\')\n        if \'Image ID\' not in previous:\n            raise ValueError(\'The comparison CSV has no image identifier.\')\n        # Never compare the new melanoma case against the older benign case\'s scores.\n        previous = previous[previous[\'Image ID\'].astype(str) == image_id].set_index(\'Pipeline\')\n    results, rows = [], []\n    for name in ORDER:\n        tf.keras.backend.clear_session()\n        gc.collect()\n        print(\'Generating every stage:\', name, flush=True)\n        result = (detailed_attention(image_path, models[name]) if name == \'Attention U-Net\'\n                  else detailed_peer(name, image_path, models[name]))\n        result.update(name=name, image_id=image_id, true_label=true_label,\n                      selection_note=selection_note, **hard_overlap(truth, result[\'mask\']))\n        result[\'prediction\'] = \'Melanoma\' if result[\'score\'] >= result[\'threshold\'] else \'Benign\'\n        if previous is not None and name in previous.index:\n            for key, column in [(\'score\', \'Model score\'), (\'dice\', \'Hard Dice (this image)\'),\n                                (\'iou\', \'Hard IoU (this image)\'), (\'threshold\', \'Decision threshold\')]:\n                if not np.isclose(result[key], previous.loc[name, column], atol=5e-4, rtol=0):\n                    raise RuntimeError(f\'{name}: {key} differs from the completed comparison. \'\n                                       \'Check the checkpoint and inference configuration.\')\n        result[\'panels\'] = stage_panels(original, truth, result)\n        result[\'no_positive_cam\'] = float(np.max(result[\'heatmap\'])) <= 1e-8\n        folder = destination / \'individual_stages\' / SLUGS[name]\n        folder.mkdir(parents=True, exist_ok=True)\n        for key, value in result[\'panels\'].items():\n            if key in (\'probability\', \'heatmap\'):\n                np.save(folder / (key + \'.npy\'), value)\n                Image.fromarray(np.uint8(np.clip(value, 0, 1) * 255)).save(folder / (key + \'.png\'))\n            else:\n                Image.fromarray(value).save(folder / (key + \'.png\'))\n        np.save(folder / \'classifier_roi_input_float32.npy\', np.asarray(result[\'roi_pixels\'], np.float32))\n        np.save(folder / \'native_gradcam_heatmap.npy\', result[\'heatmap\'])\n        if result[\'full_classifier_input\'] is not None:\n            np.save(folder / \'classifier_full_input_float32.npy\', result[\'full_classifier_input\'])\n            Image.fromarray(array_rgb(result[\'full_classifier_input\'])).save(folder / \'classifier_full_input.png\')\n        rows.append({\'Pipeline\': name, \'Image ID\': image_id, \'True label\': true_label,\n                     \'Prediction\': result[\'prediction\'], \'Model score\': result[\'score\'],\n                     \'Correct classification\': result[\'prediction\'] == true_label,\n                     \'Decision threshold\': result[\'threshold\'], \'Dice\': result[\'dice\'],\n                     \'IoU\': result[\'iou\'], \'Mask inference\': result[\'mask_method\'],\n                     \'Classifier inference\': result[\'classifier_method\'],\n                     \'ROI treatment\': result[\'roi_note\'], \'Grad-CAM target\': \'Melanoma logit\',\n                     \'No positive Grad-CAM activation\': result[\'no_positive_cam\']})\n        results.append(result)\n        tf.keras.backend.clear_session()\n        gc.collect()\n    plot_grid(results, destination, \'FULL_COMPARISON_ALL_STAGES.png\')\n    plot_grid(results, destination, \'SLIDE_A_SEGMENTATION_COMPARISON.png\', keys=KEYS[:5])\n    plot_grid(results, destination, \'SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png\', keys=KEYS[5:])\n    for result in results:\n        plot_grid([result], destination, SLUGS[result[\'name\']] + \'_ALL_STAGES.png\')\n    table = pd.DataFrame(rows)\n    table.to_csv(destination / \'all_stage_predictions.csv\', index=False)\n    manifest = {\'source_model_comparison\': dict(source_manifest or {})}\n    manifest.update(image_id=image_id, true_label=true_label,\n                    gradcam_target=\'Melanoma logit for every pipeline; Attention ROI branch\',\n                    heatmaps=\'Normalised separately within each image/model\',\n                    original_comparison=str(previous_comparison),\n                    checkpoint_paths={name: {k: str(v) for k, v in spec.items()}\n                                      for name, spec in models.items()},\n                    created_utc=datetime.now(timezone.utc).isoformat(),\n                    selection=selection_note)\n    (destination / \'full_stage_provenance.json\').write_text(json.dumps(manifest, indent=2))\n    (destination / \'READ_ME.txt\').write_text(\n        \'FULL_COMPARISON_ALL_STAGES.png contains every stage for every pipeline.\\n\'\n        \'SLIDE_A and SLIDE_B split the same figure into readable presentation panels.\\n\'\n        \'Each pipeline also has its own full-stage image and a folder of separate PNGs.\\n\'\n        \'Ground truth is the expert lesion mask, not another model prediction.\\n\'\n        \'Probability maps are the segmenter output before thresholding.\\n\'\n        \'Isolated lesion is a visualisation made by multiplying the original image by \'\n        \'the predicted mask; each model retains its original classifier ROI treatment.\\n\'\n        \'Classifier ROI panels show the actual inference input, including JPEG preprocessing \'\n        \'where applicable. Exact float32 tensors are included.\\n\'\n        \'Grad-CAM explains the melanoma class in all rows. It is not a lesion mask \'\n        \'or an indication of clinical certainty. Attention shows its ROI branch; \'\n        \'that classifier also uses the full image. Its full-image input is exported separately.\\n\'\n        \'Colours use independent normalisation. A zero map has no positive activation.\\n\'\n        f\'These figures concern selected case {image_id} (actual label: {true_label}), \'\n        \'not overall pipeline performance.\\n\'\n        \'Peer models were newly trained in the previous comparison; training splits differ.\\n\'\n        \'The supplied U-Net++ notebook implements standard U-Net.\\n\')\n    package = destination / \'Melanoma_FULL_STAGE_VISUAL_COMPARISON.zip\'\n    with zipfile.ZipFile(package, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(destination.rglob(\'*\')):\n            if path.is_file() and path != package:\n                archive.write(path, path.relative_to(destination))\n    return package, table\n'}

for name, source in MODULES.items():
    (MODULE_DIR / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(MODULE_DIR))
importlib.invalidate_caches()
for name in ('attention_pipeline_core', 'bestcase_comparison_helpers', 'full_stage_visuals'):
    sys.modules.pop(name, None)
from full_stage_visuals import resolve_saved_models, export_all_stages

models, original_comparison, model_provenance = resolve_saved_models(
    DRIVE_ROOT, COMPARISON_ROOT, PREFERRED_COMPARISON)
case = CASES[IMAGE_ID]
print('Image:', IMAGE_ID, '| Actual label:', case['label'])
print('Using checkpoints from:', original_comparison)
print('No retraining or threshold changes. No dataset download.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = COMPARISON_ROOT / 'correct_case_visuals' / (IMAGE_ID + '_' + RUN_ID)
package, metrics = export_all_stages(
    CASE_DIR / (IMAGE_ID + '.jpg'), CASE_DIR / (IMAGE_ID + '_Segmentation.png'),
    models, OUTPUT_DIR, previous_comparison=original_comparison,
    source_manifest=model_provenance, true_label=case['label'], selection_note=case['selection'])
display(metrics.round(4))
for name in ('FULL_COMPARISON_ALL_STAGES.png', 'SLIDE_A_SEGMENTATION_COMPARISON.png',
             'SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png'):
    print(name)
    display(DisplayImage(filename=str(OUTPUT_DIR / name), width=1600))
print('Finished. Your PNGs and separate stage images are saved in:', OUTPUT_DIR)
print('Downloading the image ZIP now.')
files.download(str(package))
